# Laboratory 3 — Cleaning a Noisy ECG

**CN240 Data Science for Signal Processing · SF251 Introduction to Data Science**
Week 3 · Semester 1, Academic Year 2026 · Dr Surasak Phetmanee

---

| | |
|---|---|
| **Name** | *(your name here)* |
| **Student ID** | *(your ID here)* |
| **Section** | *(your section here)* |

## What you will do today

Week 2's ECG returns — but this time it has been **deliberately damaged**: dropouts (NaN), electrode-pop
spikes, and baseline wander. You will diagnose the damage, treat it with the week-3 recipe, and **prove**
your treatment worked.

**The recipe (order matters — lecture slide 14):**
`inspect → repair holes → kill spikes → smooth → normalise → verify`

**Deliverables (due before week 4, pushed to your GitHub repository):**
1. This notebook, fully run — the whole recipe applied
2. The **decisions table** — every constant you chose, defended in one sentence
3. **SNR before vs after** — one number that proves the cleaning helped

> **Guardrail:** the QRS peaks must survive. Over-smoothing that flattens the heartbeats fails the lab.


## Setup — run these two cells first, do not edit

The second cell loads the week-2 ECG and injects the damage. It also keeps `clean_reference` —
the undamaged original — so that at the end you can measure how close your repair came.
*(Deterministic seed: everyone gets the same damage.)*


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import medfilt

plt.rcParams['figure.figsize'] = (12, 3.5)
plt.rcParams['axes.grid'] = True
print("NumPy", np.__version__, "· Pandas", pd.__version__)

In [ ]:
# --- Lab data setup: load ECG and damage it (run once, do not edit) ----------
FS = 360                                        # sampling rate, Hz
rng = np.random.default_rng(240)

try:                                            # real 5-minute MIT-BIH excerpt
    from scipy.datasets import electrocardiogram
    clean_reference = electrocardiogram()[:60 * FS]          # first 60 s
    SOURCE = "MIT-BIH (real recording)"
except Exception:                               # offline fallback: synthetic ECG
    t = np.arange(0, 60, 1 / FS)
    beat = 1.1 * np.exp(-((t % 0.85) - 0.28) ** 2 / 0.0009)
    clean_reference = (beat - 0.25 * np.exp(-((t % 0.85) - 0.38) ** 2 / 0.004)
                       + 0.03 * rng.standard_normal(t.size))
    SOURCE = "synthetic (offline fallback)"

n = clean_reference.size
t = np.arange(n) / FS
dirty = clean_reference.copy()

# 1) baseline wander — slow 0.25 Hz drift (breathing / loose electrode)
dirty += 0.35 * np.sin(2 * np.pi * 0.25 * t)

# 2) electrode-pop spikes — 40 impulses of ±3–5 mV
spike_idx = rng.choice(n, size=40, replace=False)
dirty[spike_idx] += rng.choice([-1, 1], 40) * rng.uniform(3, 5, 40)

# 3) dropouts — 25 gaps of 5–60 samples become NaN
for start in rng.choice(n - 80, size=25, replace=False):
    dirty[start : start + rng.integers(5, 60)] = np.nan

dirty = pd.Series(dirty)
print(f"ecg loaded from {SOURCE}: {n} samples = {n/FS:.0f} s at {FS} Hz")
print(f"damage injected: {dirty.isna().sum()} NaN samples, 40 spikes, baseline wander")

---
## Task 1 — Diagnose *(worked example: read it, then run it)*

Never treat before you diagnose. Count the holes, look at the damage, and know your enemy.


In [ ]:
print("NaN count   :", dirty.isna().sum())
print("NaN fraction:", f"{dirty.isna().mean():.2%}   (small fraction -> interpolation is honest)")

plt.plot(t, dirty, linewidth=0.5)
plt.xlabel("Time (s)"); plt.ylabel("Amplitude (mV)")
plt.title("The patient: dirty ECG — spikes, drift, and gaps")
plt.show()

In [ ]:
# Zoom into 5 seconds so the three kinds of damage are visible individually
start, stop = 10, 15
sl = slice(int(start * FS), int(stop * FS))
plt.plot(t[sl], dirty[sl], linewidth=0.8)
plt.xlabel("Time (s)"); plt.ylabel("Amplitude (mV)")
plt.title(f"Zoom {start}–{stop} s — can you spot a spike, a gap, and the drift?")
plt.show()

**Q1 — from the two plots:** identify one example of each damage type (give the approximate time in
seconds): a dropout, a spike, and the baseline wander.

*(your answer here)*


---
## Task 2 — Choose your constants

These three numbers **are** the lab. Choose them now, refine them as you see results, and defend each
one in the decisions table at the end. The lecture gave you the reasoning tools:

- `LIMIT_MS` — the longest gap you are willing to bridge by interpolation *(slide 6: bridge 50 ms honestly, never 5 s)*
- `KERNEL` — median filter width in samples, **must be odd** *(slide 13: just wide enough to swallow a 1-sample spike)*
- `WINDOW` — moving-average width in samples *(slide 12: well under the QRS width ≈ 80 ms × fs)*


In [ ]:
# TODO: replace None with your chosen values
LIMIT_MS = None      # e.g. somewhere between 20 and 100 (milliseconds)
KERNEL   = None      # odd number, e.g. 3–9 (samples)
WINDOW   = None      # e.g. 5–25 (samples) — remember the QRS guardrail!

if None in (LIMIT_MS, KERNEL, WINDOW):
    print("choose your three constants above before continuing")
else:
    print(f"limit = {LIMIT_MS} ms = {int(LIMIT_MS/1000*FS)} samples | "
          f"kernel = {KERNEL} | window = {WINDOW}")
    assert KERNEL % 2 == 1, "median filter kernel must be odd!" 

---
## Task 3 — Treat, one step at a time

Apply the recipe **in order**. After each step, the plot shows what that step changed — look at it
before moving on.


In [ ]:
# Step 1 — repair the holes (linear interpolation, capped)
if None in (LIMIT_MS, KERNEL, WINDOW):
    print("constants not chosen yet"); step1 = None
else:
    limit_samples = int(LIMIT_MS / 1000 * FS)
    step1 = dirty.interpolate(method="linear", limit=limit_samples)
    print("NaN remaining after repair:", step1.isna().sum(),
          " (gaps longer than your limit stay honest NaN)")
    step1 = step1.ffill().bfill()      # close any stragglers at the edges
    plt.plot(t[sl], step1[sl], linewidth=0.8)
    plt.title("after step 1 — holes bridged"); plt.xlabel("Time (s)"); plt.show()

In [ ]:
# Step 2 — kill the spikes (median filter)
if step1 is None:
    print("finish the steps above first"); step2 = None
else:
    step2 = pd.Series(medfilt(step1, kernel_size=KERNEL))
    plt.plot(t[sl], step2[sl], linewidth=0.8)
    plt.title("after step 2 — spikes gone (are the QRS peaks still there?)")
    plt.xlabel("Time (s)"); plt.show()

In [ ]:
# Step 3 — smooth the residual noise (moving average)
if step2 is None:
    print("finish the steps above first"); step3 = None
else:
    step3 = step2.rolling(window=WINDOW, center=True, min_periods=1).mean()
    plt.plot(t[sl], step3[sl], linewidth=0.8)
    plt.title("after step 3 — smoothed  (QRS guardrail check!)")
    plt.xlabel("Time (s)"); plt.show()

In [ ]:
# Step 4 — remove the baseline wander, then z-score normalise
# The wander is a very slow trend: estimate it with a LONG moving average (~1 s), subtract it.
if step3 is None:
    print("finish the steps above first"); cleaned = None
else:
    baseline = step3.rolling(window=FS, center=True, min_periods=1).mean()
    detrended = step3 - baseline
    cleaned = (detrended - detrended.mean()) / detrended.std()
    plt.plot(t[sl], cleaned[sl], linewidth=0.8)
    plt.title("after step 4 — detrended and z-scored")
    plt.xlabel("Time (s)"); plt.ylabel("z-score"); plt.show()

**Q2 — the long window paradox:** in step 4 we used a moving average with a *huge* window (1 second)
— exactly what slide 12 warned would destroy the QRS. Why is it the right tool *here*? What are we
keeping, and what are we throwing away?

*(your answer here)*


---
## Task 4 — Prove it

Three pieces of evidence: the before/after picture, the SNR number (your week-2 estimator), and —
because this lab kept the undamaged original — the correlation with the truth.


In [ ]:
def snr_db(x):
    """Week 2's quick SNR estimate (nan-aware): sample-to-sample diff as the noise proxy."""
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    noise = np.diff(x) / np.sqrt(2)
    return 10 * np.log10(np.var(x) / np.var(noise))

if cleaned is None:
    print("finish Task 3 first")
else:
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    axes[0].plot(t[sl], dirty[sl], linewidth=0.7)
    axes[0].set_title("BEFORE — dirty")
    axes[1].plot(t[sl], cleaned[sl], linewidth=0.7, color="darkred")
    axes[1].set_title("AFTER — your cleaning")
    axes[1].set_xlabel("Time (s)")
    plt.tight_layout(); plt.show()

    print(f"SNR before : {snr_db(dirty):6.1f} dB")
    print(f"SNR after  : {snr_db(cleaned):6.1f} dB")

    ref = (clean_reference - clean_reference.mean()) / clean_reference.std()
    r = np.corrcoef(cleaned, ref)[0, 1]
    print(f"correlation with the undamaged original: r = {r:.3f}   (closer to 1 = better repair)")

**Q3 — interpret your evidence:** did SNR improve, and is the correlation high? If you push `WINDOW`
up to 101, what happens to the correlation — and why does that match the lecture's over-smoothing
warning? *(Try it, then put your constants back.)*

*(your answer here)*


---
## Task 5 — The decisions table *(deliverable 2)*

| Constant | Your value | One-sentence defence |
|---|---|---|
| `LIMIT_MS` | | |
| `KERNEL` | | |
| `WINDOW` | | |

*A defence names the trade-off: e.g. “50 ms bridges electrode dropouts but refuses to invent whole
heartbeats.”*


---
## Bonus (optional) — beat the reference pipeline

Can you push the correlation higher? Ideas: detrend **before** the median filter; try a
`medfilt` second pass; tune the windows. Record your best `r` and what changed it.


In [ ]:
# Bonus playground — copy the pipeline here and experiment


---
## Before you submit (checklist)

- [ ] **Runtime → Restart and run all** passes, with all plots showing
- [ ] Every plot has time in seconds and labelled axes
- [ ] QRS peaks clearly survive in the AFTER plot
- [ ] SNR before/after printed; correlation reported
- [ ] Decisions table filled in — one honest sentence per constant
- [ ] Q1–Q3 answered · pushed to your GitHub repository

*Next week: Understanding Signals — mean, RMS, variance, peak detection and correlation, computed on
the clean ECG you just made. Assignment 1 is released and project domains are chosen — come prepared.*
